In [ ]:
# ROGII wellbore geology — 2-D candidate-grid U-Net snapshot blend (+ optional LightGBM margin)
# Inference only. Attach a Kaggle dataset holding src/*.py from this repository, the 18 checkpoints
# full_w{16,24,32}_s0_ep{7..12}.pt (src/grid_unet.py --fold 0) and margin_lgbm.txt (src/final_margin.py).
MODE = "B"   # "A" = 3-width U-Net snapshot blend, "B" = A + LightGBM margin
import sys, glob, os, numpy as np, pandas as pd, torch
SRC = sorted(glob.glob('/kaggle/input/**/grid_unet.py', recursive=True))[0].rsplit('/', 1)[0]
sys.path.insert(0, SRC)
DATA = sorted(glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True))[0].rsplit('/', 1)[0]
assert os.path.isdir(DATA + '/test'), DATA
print('code', SRC, 'data', DATA, 'cuda', torch.cuda.is_available())

In [ ]:
import tracker as lt, grid_unet as G
def read_split(split):
    wells = {}
    for f in sorted(glob.glob(f'{DATA}/{split}/*__horizontal_well.csv')):
        wid = os.path.basename(f).split('__')[0]
        h = pd.read_csv(f); tw = pd.read_csv(f'{DATA}/{split}/{wid}__typewell.csv').dropna(subset=['TVT', 'GR']).sort_values('TVT')
        p = int(h['TVT_input'].notna().sum())
        w = {k.lower(): h[k].to_numpy(float) for k in ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']}
        w['tvt'] = h['TVT'].to_numpy(float) if 'TVT' in h else np.full(len(h), np.nan)
        w.update(well_id=wid, p=p, n=len(h), tw_tvt=tw['TVT'].to_numpy(float), tw_gr=tw['GR'].to_numpy(float))
        wells[wid] = w
    return wells
wells = read_split('test'); ids = sorted(wells)
print(len(ids), 'test wells')

In [ ]:
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if dev.type == 'cpu':
    import contextlib; torch.autocast = lambda *a, **k: contextlib.nullcontext()
bank = G.WellBank(wells, ids, dev)
cks = sorted(glob.glob(f'{SRC}/full_w*_s0_ep*.pt')); print(len(cks), 'checkpoints')
acc = None
for ck in cks:
    sd = torch.load(ck, map_location=dev)
    net = G.GridUNet(sd['width']).to(dev); net.load_state_dict(sd['model']); net.eval()
    preds, _ = G.predict(net, bank)
    acc = preds if acc is None else [a + b for a, b in zip(acc, preds)]
blocks = [a / len(cks) for a in acc]

In [ ]:
def to_rows(w, off, bs=G.BS):
    p, n = w['p'], w['n']; L = len(off)
    c = p + np.arange(L) * bs + (bs - 1) / 2
    return float(w['tvt_input'][p - 1]) + np.interp(np.arange(p, n), np.r_[p - 1, c], np.r_[0.0, off])
A = {wid: to_rows(wells[wid], b) for wid, b in zip(ids, blocks)}
P = A
if MODE == 'B':
    import lightgbm as lgb, stack as S
    m = lgb.Booster(model_file=f'{SRC}/margin_lgbm.txt')
    P = {wid: A[wid] + m.predict(S.row_features(wells[wid], A[wid])) for wid in ids}

In [ ]:
rows = []
for wid in ids:
    w = wells[wid]
    for r, v in zip(range(w['p'], w['n']), P[wid]):
        rows.append((f'{wid}_{r}', float(v)))
sub = pd.DataFrame(rows, columns=['id', 'tvt'])
ss = pd.read_csv(f'{DATA}/sample_submission.csv')
sub = ss[['id']].merge(sub, on='id', how='left')
miss = sub['tvt'].isna().sum(); print('missing ids filled by carry:', miss)
if miss:
    carry = {wid: float(wells[wid]['tvt_input'][wells[wid]['p'] - 1]) for wid in ids}
    sub['tvt'] = sub['tvt'].fillna(sub['id'].str.rsplit('_', n=1).str[0].map(carry))
sub.to_csv('submission.csv', index=False); print(sub.shape); sub.head()